# Module 3.4 — Properties, Descriptors & Dataclasses

### Python Mastery · Track 3: Object-Oriented Programming & Design Patterns

Plain attributes are simple but offer no control. Properties let you run code when an attribute is read or written, while keeping the clean dotted syntax. Descriptors are the lower-level mechanism that powers properties and enables reusable attribute logic. Dataclasses remove the boilerplate of data-holding classes. This module covers all three.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Try assigning invalid values to see validation in action.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Use `@property` to compute attributes and validate assignments.
2. Add setters and deleters with `@name.setter` and `@name.deleter`.
3. Explain the descriptor protocol and write a reusable descriptor.
4. Define concise classes with `@dataclass`, including defaults and `field`.
5. Use `frozen` and `order` dataclasses and the `__post_init__` hook.

**Prerequisites:** Tracks 1 and 2, and Modules 3.1 to 3.3.

---

## Part 1 · Properties for Computed and Validated Attributes

A property looks like a plain attribute to the user but runs a method behind the scenes. Decorate a method with `@property` and access it without parentheses. This is ideal for **computed** values that depend on other data, so they always stay consistent.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        # Accessed as circle.area, not circle.area(); computed on demand.
        return 3.141592653589793 * self.radius ** 2

    @property
    def diameter(self):
        return self.radius * 2

c = Circle(5)
print("radius:", c.radius)
print("area:", round(c.area, 2))         # no parentheses
print("diameter:", c.diameter)

c.radius = 10                             # change the underlying data
print("area updates automatically:", round(c.area, 2))

## Part 2 · Setters and Validation

A property can also control assignment. Pair the getter with a setter using `@name.setter`. This lets you validate or transform a value before storing it, while users still write a simple `obj.attr = value`. By convention the real data is kept in a "private" attribute with a leading underscore.

In [ ]:
class Temperature:
    def __init__(self, celsius=0):
        self.celsius = celsius            # this goes through the setter below

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        # Validate on every assignment, including the one in __init__.
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

t = Temperature(25)
print("celsius:", t.celsius, "| fahrenheit:", t.fahrenheit)

t.celsius = 100                           # validated and stored
print("updated fahrenheit:", t.fahrenheit)

try:
    t.celsius = -300                      # rejected by the setter
except ValueError as e:
    print("rejected:", e)

A deleter, added with `@name.deleter`, runs when `del obj.attr` is used. It is less common but completes the property interface.

In [ ]:
class Profile:
    def __init__(self, nickname):
        self._nickname = nickname

    @property
    def nickname(self):
        return self._nickname

    @nickname.deleter
    def nickname(self):
        print("clearing nickname")
        self._nickname = None

p = Profile("Ace")
print("before:", p.nickname)
del p.nickname                            # triggers the deleter
print("after:", p.nickname)

## Part 3 · The Descriptor Protocol

A property works for one attribute on one class. When you need the **same** behaviour, such as a validation rule, applied to several attributes or reused across classes, a descriptor is the tool. A descriptor is a class implementing `__get__`, `__set__`, and optionally `__delete__`. Placed as a class attribute, it intercepts access to that attribute on every instance. `__set_name__` conveniently captures the attribute's name.

In [ ]:
class Positive:
    """A reusable descriptor that allows only positive numbers."""
    def __set_name__(self, owner, name):
        # Called automatically with the attribute name it was assigned to.
        self._name = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self                    # accessed on the class itself
        return getattr(instance, self._name)

    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError(f"{self._name[1:]} must be positive")
        setattr(instance, self._name, value)

class Order:
    quantity = Positive()                  # the descriptor governs this attribute
    price = Positive()                     # and this one, with no repeated code

    def __init__(self, quantity, price):
        self.quantity = quantity           # routed through Positive.__set__
        self.price = price

o = Order(3, 19.99)
print("quantity:", o.quantity, "| price:", o.price)

try:
    o.quantity = -1                        # rejected by the descriptor
except ValueError as e:
    print("rejected:", e)

## Part 4 · Dataclasses

Many classes exist mainly to hold data, and writing `__init__`, `__repr__`, and `__eq__` by hand is repetitive. The `@dataclass` decorator generates these for you from annotated fields. You declare each field with a type annotation, and the decorator does the rest.

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int
    label: str = "origin"        # a default value, like a default argument

p = Point(3, 4)
print("auto __repr__:", p)                       # Point(x=3, y=4, label='origin')
print("auto __eq__:", p == Point(3, 4))          # True, compared by field values
print("with a default overridden:", Point(1, 1, "corner"))

For fields that need a fresh mutable default (such as a list), use `field(default_factory=...)` rather than a bare `[]`, which would be shared. The `__post_init__` method runs after the generated `__init__`, which is the place for validation or derived fields.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Cart:
    owner: str
    items: list = field(default_factory=list)   # a new list per instance
    total: float = field(init=False, default=0.0)  # not a constructor argument

    def __post_init__(self):
        # Runs after __init__; compute a derived field here.
        self.total = 0.0                            # would sum real prices in practice

    def add(self, item):
        self.items.append(item)

a, b = Cart("Asha"), Cart("Ben")
a.add("book")
print("a.items:", a.items)
print("b.items independent:", b.items)              # [] -- not shared

## Part 5 · Frozen and Ordered Dataclasses

Two options extend dataclasses usefully. `frozen=True` makes instances immutable, so their fields cannot be reassigned after creation, which also makes them hashable and suitable as dictionary keys or set members. `order=True` generates comparison methods (`<`, `<=`, and so on), comparing field by field, which lets instances be sorted.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Coordinate:
    lat: float
    lon: float

c = Coordinate(18.5, 73.8)
print("frozen instance:", c)
print("usable as a dict key:", {c: "Pune"})

try:
    c.lat = 0                              # immutable: assignment is blocked
except Exception as e:
    print("cannot modify:", type(e).__name__)

@dataclass(order=True)
class Score:
    points: int
    name: str

scores = [Score(88, "Asha"), Score(92, "Ben"), Score(75, "Chen")]
print("sorted by fields:", sorted(scores))   # compares points first, then name

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A computed property (Easy)

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

r = Rectangle(3, 4)
print("area:", r.area)        # no parentheses
r.width = 5
print("area updates:", r.area)

### Example 2 — A simple dataclass (Easy)

In [ ]:
from dataclasses import dataclass

@dataclass
class Book:
    title: str
    author: str
    year: int = 2025

b = Book("Dune", "Herbert", 1965)
print(b)
print("equal:", b == Book("Dune", "Herbert", 1965))

### Example 3 — Validation through a setter (Medium)

In [ ]:
class Account:
    def __init__(self, balance=0):
        self.balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("balance cannot be negative")
        self._balance = value

a = Account(100)
a.balance = 250
print("balance:", a.balance)
try:
    a.balance = -10
except ValueError as e:
    print("rejected:", e)

### Example 4 — A dataclass with a computed total (Medium)

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Invoice:
    customer: str
    amounts: list = field(default_factory=list)
    total: float = field(init=False, default=0.0)

    def __post_init__(self):
        self.total = sum(self.amounts)

inv = Invoice("Asha", [100, 250, 75])
print(inv)
print("total:", inv.total)

### Example 5 — A reusable validating descriptor (Difficult)

In [ ]:
class Ranged:
    """Allow only values within an inclusive range."""
    def __init__(self, low, high):
        self.low = low
        self.high = high

    def __set_name__(self, owner, name):
        self._name = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self._name)

    def __set__(self, instance, value):
        if not (self.low <= value <= self.high):
            raise ValueError(f"value must be between {self.low} and {self.high}")
        setattr(instance, self._name, value)

class Settings:
    volume = Ranged(0, 100)
    brightness = Ranged(0, 100)

    def __init__(self, volume, brightness):
        self.volume = volume
        self.brightness = brightness

s = Settings(50, 80)
print("volume:", s.volume, "| brightness:", s.brightness)
try:
    s.volume = 150
except ValueError as e:
    print("rejected:", e)

### Example 6 — Frozen, ordered records used together (Difficult)

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, order=True)
class Version:
    major: int
    minor: int
    patch: int

    def __str__(self):
        return f"{self.major}.{self.minor}.{self.patch}"

versions = [Version(1, 2, 0), Version(1, 0, 5), Version(2, 0, 0), Version(1, 2, 1)]

print("sorted ascending:", [str(v) for v in sorted(versions)])
print("latest:", str(max(versions)))

# Frozen instances are hashable, so they can index a dictionary.
notes = {Version(1, 0, 0): "initial release", Version(2, 0, 0): "rewrite"}
print("note for 2.0.0:", notes[Version(2, 0, 0)])

---

## Exercises

**Exercise 1 (Easy).** Add a computed property `perimeter` to a `Square` class with a `side` attribute, returning `4 * side`. Demonstrate that it updates when `side` changes.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Define a dataclass `Person` with fields `name` (str) and `age` (int, default 0). Create two equal instances and confirm they compare equal.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Give a `Product` class a `price` property whose setter rejects negative values with a `ValueError`. Demonstrate both a valid and an invalid assignment.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Define a dataclass `Team` with `name` and a `members` list that defaults to empty using `field(default_factory=list)`. Create two teams, add a member to one, and confirm the other is unaffected.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a reusable descriptor `NonEmpty` that rejects empty strings, and use it for two attributes (`first` and `last`) of a `Name` class. Show that assigning an empty string is rejected.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
class Square:
    def __init__(self, side):
        self.side = side
    @property
    def perimeter(self):
        return 4 * self.side

sq = Square(3)
print(sq.perimeter)
sq.side = 5
print(sq.perimeter)

**Solution 2**

In [ ]:
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int = 0

print(Person("Asha", 30) == Person("Asha", 30))

**Solution 3**

In [ ]:
class Product:
    def __init__(self, price):
        self.price = price
    @property
    def price(self):
        return self._price
    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("price cannot be negative")
        self._price = value

p = Product(10)
p.price = 25
print("price:", p.price)
try:
    p.price = -5
except ValueError as e:
    print("rejected:", e)

**Solution 4**

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Team:
    name: str
    members: list = field(default_factory=list)

a = Team("Alpha")
b = Team("Beta")
a.members.append("Asha")
print("a:", a.members, "| b:", b.members)

**Solution 5**

In [ ]:
class NonEmpty:
    def __set_name__(self, owner, name):
        self._name = "_" + name
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self._name)
    def __set__(self, instance, value):
        if not value:
            raise ValueError("value cannot be empty")
        setattr(instance, self._name, value)

class Name:
    first = NonEmpty()
    last = NonEmpty()
    def __init__(self, first, last):
        self.first = first
        self.last = last

n = Name("Asha", "Patel")
print(n.first, n.last)
try:
    n.first = ""
except ValueError as e:
    print("rejected:", e)

---

## Summary and Key Points

- `@property` turns a method into a read-only attribute accessed without parentheses, ideal for computed values that stay consistent with their inputs.
- A `@name.setter` validates or transforms values on assignment while keeping simple dotted syntax; the real data is conventionally held in a leading-underscore attribute. A `@name.deleter` handles `del`.
- A descriptor (`__get__`, `__set__`, optional `__delete__`, with `__set_name__`) provides reusable attribute logic across many attributes or classes; properties are a built-in special case.
- `@dataclass` generates `__init__`, `__repr__`, and `__eq__` from annotated fields; use `field(default_factory=...)` for mutable defaults and `__post_init__` for validation or derived fields.
- `frozen=True` makes instances immutable and hashable; `order=True` adds comparison methods so instances can be sorted.

### Next module: 3.5 — Enums & Typed Named Tuples

The next module covers giving names to fixed sets of values with enums, and building lightweight typed records with typed named tuples.